<a href="https://colab.research.google.com/github/yaesur/business_python/blob/main/%EA%B5%90%ED%86%B5%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install geopy

import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import time
import re
from google.colab import files

파일 설정

In [18]:
input_file = "23년 2차.xlsx" #파일명만 변경

def load_data(file_path):

    xl = pd.ExcelFile(file_path)
    sheet_name = xl.sheet_names[0]
    for s in xl.sheet_names:
        if '주택목록' in s or 'Sheet1' in s:
            sheet_name = s
            break


    df_temp = pd.read_excel(file_path, sheet_name=sheet_name)


    skip_idx = 0
    for i, row in df_temp.iterrows():
        if any('소재지 주소' in str(val) for val in row.values):
            skip_idx = i + 1
            break

    return pd.read_excel(file_path, sheet_name=sheet_name, skiprows=skip_idx)


try:
    df_houses = load_data(input_file)
    print({len(df_houses)})
except Exception as e:
    print(f"오류: {e}")

# 지하철 정보 로드
try:
    df_subway_all = pd.read_csv("서울시 역사마스터 정보.csv", encoding='cp949')
except:
    df_subway_all = pd.read_csv("서울시 역사마스터 정보.csv", encoding='utf-8-sig')

{581}


분석환경 설정

In [19]:
subway_counts = df_subway_all.groupby('역사명')['호선'].nunique().to_dict()
geolocator = Nominatim(user_agent="seoul_sh_universal_v3")

def analyze_logic(row):

    addr_col = [c for c in row.index if '주소' in str(c)]
    if not addr_col: return ["컬럼오류", "N/A", "N/A", "N/A"]

    addr = str(row[addr_col[0]])
    if addr == 'nan' or not addr: return ["주소없음", "N/A", "N/A", "N/A"]

    clean_addr = re.sub(r'\(.*\)', '', addr)
    clean_addr = re.sub(r'\d+호', '', clean_addr).split(',')[0].strip()

    try:
        location = geolocator.geocode(clean_addr)
        if not location:

            gu_col = [c for c in row.index if '자치구' in str(c)]
            name_col = [c for c in row.index if '주택명' in str(c)]
            if gu_col and name_col:
                location = geolocator.geocode(f"서울특별시 {row[gu_col[0]]} {row[name_col[0]]}")

        if not location: return ["좌표실패", "N/A", "N/A", "N/A"]

        house_pos = (location.latitude, location.longitude)
        df_st_unique = df_subway_all.drop_duplicates('역사명').copy()
        df_st_unique['dist'] = df_st_unique.apply(
            lambda x: geodesic(house_pos, (x['위도'], x['경도'])).km, axis=1
        )

        nearest = df_st_unique.loc[df_st_unique['dist'].idxmin()]
        st_name = nearest['역사명']
        dist_km = nearest['dist']
        walking_min = round(dist_km * 15)

        n_lines = subway_counts.get(st_name, 1)
        grade = "Grade 4+" if n_lines >= 4 else f"Grade {n_lines}"

        return [st_name, grade, f"{round(dist_km, 2)}km", f"{walking_min}분"]
    except:
        return ["오류", "오류", "오류", "오류"]


분석 실행 및 파일 다운

In [23]:
results = []

for idx, row in df_houses.iterrows():
    results.append(analyze_logic(row))
    time.sleep(1.2)
    if (idx + 1) % 50 == 0:
      print(f"{idx + 1} / {len(df_houses)} 건 완료")

res_cols = ['인근역', '교통등급', '거리(km)', '도보시간(분)']
df_res = pd.DataFrame(results, columns=res_cols, index=df_houses.index)
final_df = pd.concat([df_houses, df_res], axis=1)

output_name = f"분석결과_{input_file.split('.')[0]}.xlsx"
final_df.to_excel(output_name, index=False)

print(f"\n 분석 완료")
files.download(output_name)

현재 50개 데이터 분석 중


KeyboardInterrupt: 